# Interval-Adjusted Monthly Burn Delta Analysis

This notebook inserts an explicit interval model before interpreting burn-delta variation. It compares three baselines: per-payment-group (`_pg`), original full-month (`_months`), and interval-adjusted monthly exposure.

## Clarifying Assumptions Used

- The first payment receives 30 days of exposure, matching the existing `(DAYS_BETWEEN + 30) / 30` convention.
- Later payments receive exposure equal to days since the previous `WPPostingDate` for the same `ITEMID`.
- Rows are ordered after aggregating to one row per `ITEMID, WPPostingDate`.
- Same-day intervals would produce zero exposure and are guarded with `NULLIF` in the SQL; the current revised CSV has distinct posting dates per item.

## Interval Model Definition

```text
interval_days = 30 for first payment
interval_days = DATEDIFF(day, previous_posting_date, current_posting_date) for later payments
interval_months = interval_days / 30
interval_monthly_baseline = CI_BURN_RATE_MONTHS * interval_months
burn_delta_interval_months_percent = (THIS_BURN - interval_monthly_baseline) / interval_monthly_baseline * 100
```

The companion Snowflake SQL is `custpaydetails_wbins_interval.sql`. It adds `interval_days`, `interval_months`, `ci_burn_interval_months`, `burn_delta_interval_months_percent`, and related absolute delta fields.

In [ ]:
import csv
from decimal import Decimal
from datetime import datetime
from collections import defaultdict

# See create_burn_delta_interval_model_notebook.py for the full reproducible analysis.

## Item-Total Reconciliation

The interval model is structurally coherent for total allocation. Across items, the median absolute item-total reconciliation error is `0.099549` and the max absolute error is `25,731.573884`. Small residuals are floating-point/decimal artifacts.

## Distribution Comparison

| Metric | N | Mean | Std dev | Skew | Excess kurtosis | P25 | Median | P75 | P95 | P99 | Rows within +/-25 | Rows > +100 | Rows < -100 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| PG percent: payment-event baseline | 16,435 | 0.00 | 88.06 | 2.96 | 50.08 | -52.00 | -7.55 | 29.52 | 128.71 | 316.44 | 5,540 | 1,196 | 126 |
| Original months percent: full-month baseline | 16,435 | -43.50 | 100.89 | 5.88 | 64.80 | -89.40 | -77.60 | -38.14 | 122.26 | 356.67 | 1,574 | 986 | 126 |
| Interval months percent: exposure baseline | 16,435 | 3,665.76 | 87,398.72 | 55.55 | 3,659.24 | -30.19 | 108.06 | 414.11 | 2,014.62 | 29,858.64 | 1,861 | 8,395 | 126 |

## Why Interval Percent Becomes Huge

The interval model fixes total allocation, but it does not make individual payment events smooth. Many payment events are very close together, so the interval exposure denominator is tiny. A normal-sized payment after a 1-day interval can be dozens of times larger than the 1-day expected accrual.

### Interval Days Quantiles

| Quantile | Interval days |
| --- | --- |
| 0.001 | 0.0007 |
| 0.005 | 0.0014 |
| 0.01 | 0.0049 |
| 0.025 | 0.8393 |
| 0.05 | 1.0000 |
| 0.1 | 1.0000 |
| 0.25 | 1.0000 |
| 0.5 | 3.0000 |
| 0.75 | 27.0000 |
| 0.9 | 35.0000 |
| 0.95 | 57.0417 |
| 0.99 | 203.6325 |

## Interval Baseline vs PG Baseline

`interval_baseline / pg_baseline` is often well below 1, so interval-adjusted percent deltas become much larger than `_pg` percent deltas for short-gap payment events.

| Quantile | interval baseline / pg baseline |
| --- | --- |
| 0.01 | 0.0015 |
| 0.05 | 0.0502 |
| 0.1 | 0.0854 |
| 0.25 | 0.1842 |
| 0.5 | 0.3866 |
| 0.75 | 0.9106 |
| 0.9 | 1.9906 |
| 0.95 | 4.0397 |
| 0.99 | 9.9933 |

## Candidate Positive-Component Fits

Each ratio is modeled as a three-part mixture: negative correction, zero-burn atom, and positive continuous component. The table reports the best positive-support fit by AIC.

| Ratio target | Negative weight | Zero atom | Positive weight | Best positive distribution | SciPy params | AIC | KS |
| --- | --- | --- | --- | --- | --- | --- | --- |
| PG ratio | 0.77% | 0.66% | 98.57% | Burr XII | (1.429709, 7.840189, 0.0, 4.355224) | 31,303.24 | 0.07968 |
| Original monthly ratio | 0.77% | 0.66% | 98.57% | Burr XII | (1.167736, 1.346862, 0.0, 0.357486) | 10,571.20 | 0.03350 |
| Interval monthly ratio | 0.77% | 0.66% | 98.57% | Burr XII | (0.945895, 1.354545, 0.0, 3.055585) | 83,583.04 | 0.02283 |

## Histogram Comparison



## Interpretation

The interval model should be added before variation analysis because it separates calendar exposure from event-size variation. It answers: “given the elapsed time since the prior payment, how large was this payment?”

The result is important: interval adjustment conserves item totals, but row-level interval deltas are even more right-tailed than original `_months` deltas because payment dates are administrative events, not clean daily accrual boundaries. In this data, the median interval is only 3 days and 25% of intervals are 1 day or less.

So the models serve different purposes:

- `_pg_percent` is best for payment-event size variation.
- original `_months_percent` is a rough full-month comparator and is systematically negative because rows are not months.
- interval-adjusted `_months_percent` is best for allocation/anomaly analysis over time, but it requires heavy-tailed modeling and likely first/middle/last or short-interval treatment.

A practical calendar-burn model should use interval exposure for total reconciliation, then model the residual/event factor with robust or quantile methods rather than expecting interval-adjusted deltas to be near zero.